# Notebook 1B — Alur PCV + Segmentasi/Masking IS-Net (Pre-processing IS-Net → Segmentasi IS-Net → Ekstraksi Fitur → Bounding Box Manual)

Notebook Google Colab ini merupakan adaptasi dari **Notebook 1 (`1_pcv_klasik.ipynb`)** dengan **tahap segmentasi/masking langsung menggunakan model IS-Net (`isnet-general-use` via `rembg` ONNX)** — **tanpa aturan `if-else` adaptif maupun metode heuristik klasik** — dan format ekspor CSV/JSON/ZIP tetap **100% kompatibel dengan `5_inferensi_ssd_dari_output_nb1.ipynb`**.

### Alur Tahapan yang Divisualisasikan
1. **Input Citra** — Google Drive (`SOURCE_MODE = "drive"`), Unggah Langsung (`"upload"`), atau Lokal (`"local"` → `data/helper`).
2. **Pre-processing Input IS-Net (`1024×1024`)** — Resize LANCZOS `1024×1024`, normalisasi skala `0..1` (`/ max`), & pengurangan mean ImageNet `(0.485, 0.456, 0.406)` → tensor `(1, 3, 1024, 1024)` (serta Grayscale sederhana untuk ekstraksi fitur HOG/GLCM).
3. **Segmentasi / Masking IS-Net (`isnet-general-use`) + Bedah Proses Internal** —
   - Otomatis berjalan di **GPU (`CUDAExecutionProvider`)** bila GPU tersedia (tanpa batas VRAM), dengan fallback CPU otomatis.
   - Membedah proses di dalam graf ONNX IS-Net: `conv_in` → Encoder `1–6` (`RSU-7`..`RSU-4F`) → Decoder `5–1` (`RSU-4F`..`RSU-7`) → side outputs `d1..d6`.
   - Pascaproses `d1`: normalisasi min-max → resize LANCZOS ke ukuran asli → threshold `alpha > 127` → kontur terbesar + tutup lubang (*hole filling*) + verifikasi langsung terhadap keluaran `rembg` asli.
4. **Penyempurnaan Mask, Kontur, & Ekstraksi Fitur** — Geometri Kontur (`Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio`, `Extent`, `Solidity`, `Circularity`, `Hu Moments`), **HOG**, **Tekstur GLCM**, & **Warna RGB/HSV**.
5. **Hasil Akhir & Ekspor Batch Paralel (Penuh RAM & GPU VRAM)** — **Bounding Box manual tanpa label kelas** (`per_gambar/*.bbox.jpg` & `.json`), `07_ringkasan_segmentasi_isnet.csv`, `07_hasil_bounding_box_semua.json`, dan paket `nb1b_pcv_isnet_outputs.zip`.

## Cell 1 — Setup Library, Akselerasi GPU/RAM Penuh (`rembg` & `onnx`), & Folder Output Presentasi

In [ ]:
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Deteksi GPU NVIDIA (tanpa membatasi RAM atau VRAM GPU)
HAS_CUDA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
)

# Pasang rembg[gpu] saat GPU tersedia agar IS-Net berjalan di VRAM GPU, atau rembg[cpu] jika tanpa GPU
rembg_pkg = "rembg[gpu]" if HAS_CUDA_GPU else "rembg[cpu]"
need_rembg_install = importlib.util.find_spec("rembg") is None
if HAS_CUDA_GPU and not need_rembg_install:
    try:
        import onnxruntime as _ort_check
        if "CUDAExecutionProvider" not in _ort_check.get_available_providers():
            need_rembg_install = True
    except Exception:
        need_rembg_install = True

missing_pkgs = []
if need_rembg_install:
    missing_pkgs.append(rembg_pkg)
if importlib.util.find_spec("onnx") is None:
    missing_pkgs.append("onnx")
if importlib.util.find_spec("skimage") is None:
    missing_pkgs.append("scikit-image")
if missing_pkgs:
    print("Memasang dependensi:", ", ".join(missing_pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from PIL import Image, ImageOps
from rembg import new_session
from skimage.feature import graycomatrix, graycoprops, hog

# Biarkan ONNX Runtime menemukan pustaka CUDA/cuDNN sistem/PyTorch bila tersedia (tanpa limit VRAM)
if HAS_CUDA_GPU and hasattr(ort, "preload_dlls"):
    try:
        ort.preload_dlls()
    except Exception:
        pass

AVAILABLE_PROVIDERS = ort.get_available_providers()
ORT_PROVIDERS = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if (HAS_CUDA_GPU and "CUDAExecutionProvider" in AVAILABLE_PROVIDERS)
    else ["CPUExecutionProvider"]
)

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb1b_pcv_isnet") if IN_COLAB else Path("./outputs_presentasi/nb1b_pcv_isnet")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


ram_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3)) if hasattr(os, "sysconf") else 0.0
gpu_info = "Tidak ada GPU (mode CPU)"
if HAS_CUDA_GPU:
    q = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if q.returncode == 0 and q.stdout.strip():
        gpu_info = q.stdout.strip().replace("\n", " | ")

print(f"Siap! Colab={IN_COLAB} | OpenCV={cv2.__version__} | CPU={os.cpu_count()} core | RAM={ram_gb:.1f} GB (tanpa batas)")
print(f"GPU   : {gpu_info} (tanpa batas VRAM)")
print(f"ONNX  : providers aktif = {ORT_PROVIDERS}")
print(f"Output: {OUT_DIR}")

## Cell 2 — Muat Gambar Uji (Google Drive `ferqueue/gambar`, Upload Langsung, atau Lokal)

* Di Google Colab, default sudah mengarah ke Google Drive **`MyDrive/ferqueue/gambar`** (`SOURCE_MODE = "drive"`).
* Cell 2 menampilkan **4 gambar acak** (`DISPLAY_SAMPLE_COUNT = 4`) khusus agar tampilan preview plot di Colab tetap rapi (bukan pembatasan RAM).
* Ubah `ACTIVE_INDEX` (`0`–`3`) untuk memilih sampel utama yang dibedah detail di Cell 3–7. Di **Cell 8**, seluruh gambar di folder diproses secara paralel memanfaatkan penuh RAM & VRAM GPU.

In [ ]:
SOURCE_MODE = "drive"  # Pilihan: "drive" | "upload" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/gambar"
LOCAL_IMAGE_DIR = "data/helper"
DISPLAY_SAMPLE_COUNT = 4  # Jumlah gambar acak yang ditampilkan di Colab
SAMPLE_SEED = None        # None = acak tiap run | isi angka (mis. 42) agar sampel acak selalu sama
ACTIVE_INDEX = 0          # Indeks sampel (0..3) yang dianalisis detail pada Cell 3-7

IMAGE_EXTS = {".jpg", ".jpeg", ".jfif", ".png", ".webp", ".bmp"}
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

image_paths = []
if SOURCE_MODE == "upload" and IN_COLAB:
    print("Silakan pilih 1 atau lebih gambar kendaraan...")
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif SOURCE_MODE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path(DRIVE_IMAGE_DIR)
    if not src_dir.is_dir():
        raise FileNotFoundError(f"Folder Google Drive tidak ditemukan: {src_dir}")
    image_paths = sorted(p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
else:
    src_dir = Path(LOCAL_IMAGE_DIR)
    if src_dir.is_dir():
        image_paths = sorted(p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

if not image_paths:
    raise FileNotFoundError(
        "Belum ada gambar yang dimuat. Unggah minimal 1 gambar JPG/PNG atau periksa path folder."
    )


def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(ImageOps.exif_transpose(img).convert("RGB"), dtype=np.uint8)


rng = random.Random(SAMPLE_SEED)
sample_k = min(DISPLAY_SAMPLE_COUNT, len(image_paths))
sample_paths = rng.sample(image_paths, sample_k) if len(image_paths) > sample_k else list(image_paths)
sample_path_set = set(sample_paths)
sample_images = [(p.name, load_rgb(p)) for p in sample_paths]

ACTIVE_INDEX = min(ACTIVE_INDEX, len(sample_images) - 1)
active_name, img_rgb = sample_images[ACTIVE_INDEX]

print(f"Total gambar ditemukan di folder : {len(image_paths)} gambar")
print(f"Menampilkan {len(sample_images)} gambar acak di Colab (semua {len(image_paths)} gambar akan diproses di Cell 8):")
for idx, (name, arr) in enumerate(sample_images):
    marker = " <-- AKTIF (Cell 3-7)" if idx == ACTIVE_INDEX else ""
    print(f"  [{idx}] {name} | ukuran={arr.shape[1]}x{arr.shape[0]} px | dtype={arr.dtype}{marker}")

cols = min(len(sample_images), 4)
rows = math.ceil(len(sample_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows))
axes = np.atleast_1d(axes).ravel()
for i, ax in enumerate(axes):
    if i < len(sample_images):
        name, arr = sample_images[i]
        ax.imshow(arr)
        tag = " [AKTIF]" if i == ACTIVE_INDEX else ""
        ax.set_title(f"[{i}] {name[:24]}{tag}\n{arr.shape[1]}x{arr.shape[0]} px", fontsize=9)
    ax.axis("off")
fig.suptitle(f"Sampel {len(sample_images)} Gambar Acak dari Total {len(image_paths)} Gambar", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "00_daftar_gambar_uji.png")
plt.show()

## Cell 3 — Tahap 1: Pre-processing Input IS-Net (`1024×1024`)

Karena segmentasi/masking langsung menggunakan **IS-Net (`isnet-general-use`)** tanpa aturan `if-else`, citra RGB utuh disiapkan melalui 4 langkah pre-processing standar IS-Net:
1. **Resize ke `1024×1024`** (interpolasi `LANCZOS`).
2. **Normalisasi skala ke `0..1`** (dibagi nilai maksimum piksel).
3. **Pengurangan mean ImageNet `(0.485, 0.456, 0.406)`** pada masing-masing kanal R, G, B.
4. **Susun dimensi `HWC → NCHW`** menjadi tensor `(1, 3, 1024, 1024)` bertipe `float32`.

*(Selain itu disiapkan `img_gray` sederhana khusus untuk ekstraksi fitur HOG dan tekstur GLCM pada Cell 6.)*

In [ ]:
UKURAN_ISNET = 1024
MEAN_ISNET = (0.485, 0.456, 0.406)
STD_ISNET = (1.0, 1.0, 1.0)


def praproses_isnet(rgb: np.ndarray) -> tuple:
    """Pre-processing masukan model IS-Net: Resize LANCZOS 1024x1024 -> Skala 0..1 -> Kurangi mean ImageNet."""
    pil_img = Image.fromarray(rgb)
    kecil = pil_img.resize((UKURAN_ISNET, UKURAN_ISNET), Image.Resampling.LANCZOS)
    arr = np.asarray(kecil, dtype=np.float32)
    skala = arr / max(float(arr.max()), 1e-6)  # nilai 0..1
    norm = (skala - np.array(MEAN_ISNET, np.float32)) / np.array(STD_ISNET, np.float32)
    tensor = norm.transpose(2, 0, 1)[None].astype(np.float32)  # (1, 3, 1024, 1024)
    return kecil, skala, tensor


pil_kecil, rgb_skala, tensor_isnet = praproses_isnet(img_rgb)
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)  # Grayscale sederhana untuk ekstraksi fitur HOG/GLCM

nama_kanal = ["R", "G", "B"]
print("=== PRE-PROCESSING MASUKAN IS-NET ===")
print(f"Foto masuk                  : {img_rgb.shape[1]} x {img_rgb.shape[0]} piksel")
print(f"1. Resize                   : {UKURAN_ISNET} x {UKURAN_ISNET} piksel (interpolasi LANCZOS)")
print(f"2. Bagi nilai maksimum      : {rgb_skala.min():.3f} sampai {rgb_skala.max():.3f}")
for c in range(3):
    print(f"3. Kurangi mean kanal {nama_kanal[c]}      : mean {MEAN_ISNET[c]} -> {tensor_isnet[0, c].min():.3f} sampai {tensor_isnet[0, c].max():.3f}")
print(f"4. Tensor masukan model     : bentuk {tensor_isnet.shape}, tipe {tensor_isnet.dtype}")

fig_is, axes_is = plt.subplots(1, 6, figsize=(22, 4.0))
axes_is[0].imshow(img_rgb);       axes_is[0].set_title(f"Foto masuk\n{img_rgb.shape[1]} x {img_rgb.shape[0]}")
axes_is[1].imshow(pil_kecil);     axes_is[1].set_title("1. Resize\n1024 x 1024")
axes_is[2].imshow(rgb_skala);     axes_is[2].set_title("2. Dibagi nilai maks\n(skala 0..1)")
for c in range(3):
    axes_is[3 + c].imshow(tensor_isnet[0, c], cmap="gray")
    axes_is[3 + c].set_title(f"3. Kanal {nama_kanal[c]} - {MEAN_ISNET[c]}\n(tensor masukan)")
for ax in axes_is:
    ax.axis("off")
fig_is.suptitle(f"Tahap 1: Pre-processing Masukan IS-Net — {active_name}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.91])
save_fig(fig_is, "01_preprocessing_isnet.png")
plt.show()

## Cell 4 — Tahap 2: Segmentasi / Masking IS-Net (`isnet-general-use`) + Bedah Proses Internal

Sel ini menjalankan segmentasi/masking langsung menggunakan **IS-Net (`isnet-general-use`, lewat sesi ONNX `rembg`)**:
1. **Bedah Proses Internal IS-Net** (`INSPECT_ISNET_STAGES = True`):
   - `conv_in` (`3×3` stride 2) → Encoder `1–6` (`RSU-7` s/d `RSU-4F`) → Decoder `5–1` (`RSU-4F` s/d `RSU-7`) → peta objek multi-skala `d1..d6` (setelah sigmoid).
2. **Pascaproses Mask IS-Net**:
   - Normalisasi min-max `d1` → resize LANCZOS ke ukuran asli → threshold `alpha > 127` → ambil kontur eksternal terbesar + tutup lubang (*hole filling*) → verifikasi langsung dengan `session_isnet.predict(pil)` asli.
3. **Ringkasan Visual Segmentasi IS-Net**:
   - Input RGB → Peta Probabilitas Alpha → Mask Biner Bersih → Overlay Foreground.

In [ ]:
ISNET_ALPHA_THRESH = 127     # Ambang biner kanal alpha IS-Net (0..255)
INSPECT_ISNET_STAGES = True  # True = tampilkan visualisasi bedah layer internal Encoder/Decoder IS-Net & d1..d6
WARNA_LATAR = np.array([127, 127, 127], dtype=np.uint8)

if "session_isnet" not in globals():  # model ONNX dimuat sekali saja (otomatis GPU jika tersedia, tanpa batas VRAM)
    print(f"Memuat model IS-Net (isnet-general-use) dengan providers={ORT_PROVIDERS}...")
    session_isnet = new_session("isnet-general-use", providers=ORT_PROVIDERS)
print("Sesi IS-Net siap | Provider aktif:", session_isnet.inner_session.get_providers())


def jalankan_isnet(tensor: np.ndarray, session) -> list:
    nama_masukan = session.inner_session.get_inputs()[0].name
    return session.inner_session.run(None, {nama_masukan: tensor})


def peta_ke_mask(keluaran_utama: np.ndarray, ukuran_asli: tuple, alpha_thresh: int = ISNET_ALPHA_THRESH) -> tuple:
    pred = np.squeeze(keluaran_utama)
    pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)  # min-max -> 0..1
    peta = Image.fromarray((pred * 255).astype("uint8"))
    alpha_ = np.asarray(peta.resize(ukuran_asli, Image.Resampling.LANCZOS))  # ukuran_asli = (W, H)
    return pred, alpha_, (alpha_ > alpha_thresh).astype(np.uint8)


def perbaiki_mask(mask_raw: np.ndarray) -> tuple:
    """Ambil komponen kontur eksternal terbesar, tutup lubang internal, dan hitung statistik kebersihan mask."""
    jumlah_komponen = max(0, int(cv2.connectedComponents(mask_raw)[0]) - 1)  # dikurangi 1 (latar)
    kontur_semua, _ = cv2.findContours(mask_raw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not kontur_semua:
        mask_kosong = np.zeros_like(mask_raw, dtype=np.uint8)
        info = {"komponen": 0, "lubang_terisi": 0, "noda_dibuang": 0}
        return mask_kosong, None, info
    kontur_ = max(kontur_semua, key=cv2.contourArea)  # objek utama = kontur terbesar
    mask_final_01 = np.zeros_like(mask_raw, dtype=np.uint8)
    cv2.drawContours(mask_final_01, [kontur_], -1, 1, thickness=-1)  # isi penuh (menutup lubang)
    info = {
        "komponen": int(jumlah_komponen),
        "lubang_terisi": int(((mask_final_01 == 1) & (mask_raw == 0)).sum()),
        "noda_dibuang": int(((mask_final_01 == 0) & (mask_raw == 1)).sum()),
    }
    return (mask_final_01 * 255).astype(np.uint8), kontur_, info


def bbox_manual(mask: np.ndarray) -> tuple:
    """Hitung bounding box (x, y, w, h) secara manual dari piksel mask > 0."""
    ys, xs = np.where(mask > 0)
    if xs.size == 0 or ys.size == 0:
        return 0, 0, 0, 0
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return x0, y0, x1 - x0 + 1, y1 - y0 + 1


def peta_tampil(k: np.ndarray):
    """Ubah satu keluaran tensor model menjadi gambar 2D 0..1 untuk visualisasi."""
    p = np.squeeze(k)
    if p.ndim == 3:  # berkanal banyak -> rata-rata kanal
        p = p.mean(axis=0)
    if p.ndim != 2:
        return None
    return (p - p.min()) / (p.max() - p.min() + 1e-8)


def make_overlay(rgb: np.ndarray, mask: np.ndarray, color=(40, 220, 90), alpha=0.45) -> np.ndarray:
    overlay = rgb.copy().astype(np.float32)
    tint = np.zeros_like(overlay)
    tint[:, :] = color
    m = mask > 0
    overlay[m] = (1 - alpha) * overlay[m] + alpha * tint[m]
    return overlay.astype(np.uint8)


NAMA_TAHAP_ISNET = {
    "conv_in": "Konvolusi masukan\n(3x3, stride 2)",
    "stage1": "Encoder 1 (RSU-7)", "stage2": "Encoder 2 (RSU-6)", "stage3": "Encoder 3 (RSU-5)",
    "stage4": "Encoder 4 (RSU-4)", "stage5": "Encoder 5 (RSU-4F)", "stage6": "Encoder 6 (RSU-4F)",
    "stage5d": "Decoder 5 (RSU-4F)", "stage4d": "Decoder 4 (RSU-4)", "stage3d": "Decoder 3 (RSU-5)",
    "stage2d": "Decoder 2 (RSU-6)", "stage1d": "Decoder 1 (RSU-7)",
}
POLA_TAHAP_ISNET = re.compile(r"^(conv_in|stage\d+d?)$")


def cari_model_onnx(session) -> str:
    kandidat = [
        getattr(session.inner_session, "_model_path", None),
        os.path.join(
            os.environ.get("U2NET_HOME", os.path.join(os.path.expanduser("~"), ".u2net")),
            "isnet-general-use.onnx",
        ),
    ]
    for k in kandidat:
        if isinstance(k, str) and os.path.isfile(k):
            return k
    raise FileNotFoundError("Berkas model IS-Net (.onnx) tidak ditemukan.")


def kelompokkan_tahap(daftar_node):
    """Kelompokkan node graf menurut nama tahap; ambil keluaran node terakhir tiap tahap."""
    urutan, terakhir = [], {}
    for node in daftar_node:
        if node.op_type == "Constant":
            continue
        bagian = node.name.split("/")  # contoh: /stage1/rebnconvin/conv_s1/Conv
        if len(bagian) >= 3 and bagian[0] == "" and POLA_TAHAP_ISNET.match(bagian[1]):
            kunci = bagian[1]
            if kunci not in terakhir:
                urutan.append(kunci)
            terakhir[kunci] = node.output[0]
    return urutan, terakhir


def siapkan_sesi_tahap(session):
    """Salinan graf ONNX IS-Net yang mengeluarkan output tiap tahap (file asli tidak diubah)."""
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper

    model = onnx.load(cari_model_onnx(session))
    urutan, terakhir = kelompokkan_tahap(model.graph.node)
    if not urutan:
        raise RuntimeError("Nama node pada model tidak memuat nama tahap (stage1, stage2, ...).")
    nama_tensor = [terakhir[k] for k in urutan]
    sudah = {o.name for o in model.graph.output}
    for nama in nama_tensor:
        if nama not in sudah:
            model.graph.output.append(helper.make_tensor_value_info(nama, TensorProto.FLOAT, None))
    jalur = os.path.join(tempfile.gettempdir(), "isnet_tahap.onnx")
    onnx.save(model, jalur)
    sesi = ort.InferenceSession(jalur, providers=ORT_PROVIDERS)
    return sesi, urutan, nama_tensor


def segment_isnet(rgb: np.ndarray, session, alpha_thresh: int = ISNET_ALPHA_THRESH) -> dict:
    """Jalankan segmentasi/masking IS-Net penuh pada citra RGB (H, W, 3) dan kembalikan alpha, mask, kontur, & info."""
    h, w = rgb.shape[:2]
    _, _, tensor = praproses_isnet(rgb)
    keluaran = jalankan_isnet(tensor, session)
    peta_prob, alpha, mask_raw = peta_ke_mask(keluaran[0], (w, h), alpha_thresh=alpha_thresh)
    mask_final, kontur, info_mask = perbaiki_mask(mask_raw)
    kotak = bbox_manual(mask_final)
    return {
        "tensor": tensor,
        "keluaran_model": keluaran,
        "peta_prob": peta_prob,
        "alpha": alpha,
        "mask_raw": (mask_raw * 255).astype(np.uint8),
        "mask": mask_final,
        "kontur": kontur,
        "info_mask": info_mask,
        "bbox": kotak,
    }


# Jalankan segmentasi/masking IS-Net pada gambar aktif
isnet_active = segment_isnet(img_rgb, session_isnet, alpha_thresh=ISNET_ALPHA_THRESH)
keluaran_model = isnet_active["keluaran_model"]

# --- 2a & 2b. Bedah proses internal IS-Net (Encoder/Decoder + Side Outputs d1..d6) ---
if INSPECT_ISNET_STAGES:
    print("\n=== 2a. BEDAH PROSES INTERNAL MODEL IS-NET ===")
    print(f"Jumlah keluaran tensor IS-Net: {len(keluaran_model)}")
    for idx_k, k in enumerate(keluaran_model, 1):
        print(f"  keluaran {idx_k:>2}: bentuk {k.shape}")

    try:
        if "sesi_tahap_isnet" not in globals():
            sesi_tahap_isnet, urutan_tahap_isnet, tensor_tahap_isnet = siapkan_sesi_tahap(session_isnet)
        nama_in = sesi_tahap_isnet.get_inputs()[0].name
        hasil_tahap = sesi_tahap_isnet.run(tensor_tahap_isnet, {nama_in: isnet_active["tensor"]})

        kolom_t = 6
        baris_t = (len(hasil_tahap) + kolom_t - 1) // kolom_t
        fig_t, axes_t = plt.subplots(baris_t, kolom_t, figsize=(22, 4.0 * baris_t), squeeze=False)
        for ax in axes_t.ravel():
            ax.axis("off")
        for ax, kunci, h_out in zip(axes_t.ravel(), urutan_tahap_isnet, hasil_tahap):
            gambar_t = peta_tampil(h_out)
            if gambar_t is not None:
                ax.imshow(gambar_t, cmap="gray")
            p_sq = np.squeeze(h_out)
            info_b = f"{p_sq.shape[0]} kanal, {p_sq.shape[-2]}x{p_sq.shape[-1]}" if p_sq.ndim == 3 else f"{p_sq.shape[0]}x{p_sq.shape[1]}"
            ax.set_title(f"{NAMA_TAHAP_ISNET.get(kunci, kunci)}\n{info_b}", fontsize=9.5)
        fig_t.suptitle(f"Tahap 2a: Output Tiap Tahap di Dalam IS-Net (Encoder-Decoder RSU) — {active_name[:32]}", fontsize=13.5, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        save_fig(fig_t, "02a_isnet_tahap_internal.png")
        plt.show()
    except Exception as exc:
        print("Output antar-tahap ONNX tidak dapat dibaca langsung:", repr(exc))
        fitur_model = keluaran_model[len(keluaran_model) // 2:]
        nama_fitur = ["Fitur decoder 1", "Fitur decoder 2", "Fitur decoder 3", "Fitur decoder 4", "Fitur decoder 5", "Fitur encoder 6"]
        fig_t, axes_t = plt.subplots(1, len(fitur_model), figsize=(4 * len(fitur_model), 4.0), squeeze=False)
        for idx_f, (ax, h_out) in enumerate(zip(axes_t[0], fitur_model)):
            gambar_t = peta_tampil(h_out)
            if gambar_t is not None:
                ax.imshow(gambar_t, cmap="gray")
            ax.set_title(f"{nama_fitur[idx_f] if idx_f < len(nama_fitur) else f'Fitur {idx_f + 1}'}\n{np.squeeze(h_out).shape}", fontsize=9.5)
            ax.axis("off")
        fig_t.suptitle(f"Tahap 2a: Fitur Internal IS-Net — {active_name[:32]}", fontsize=13.5, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.91])
        save_fig(fig_t, "02a_isnet_tahap_internal.png")
        plt.show()

    # Side outputs d1..d6
    n_sisi = min(6, len(keluaran_model))
    fig_d, axes_d = plt.subplots(1, n_sisi, figsize=(3.8 * n_sisi, 4.0), squeeze=False)
    for idx_s, (ax, k_out) in enumerate(zip(axes_d[0], keluaran_model[:n_sisi]), 1):
        gambar_s = peta_tampil(k_out)
        if gambar_s is not None:
            ax.imshow(gambar_s, cmap="gray")
        tag_d = " (dipakai hasil akhir)" if idx_s == 1 else ""
        sq = np.squeeze(k_out)
        ax.set_title(f"d{idx_s}{tag_d}\n{sq.shape[0]}x{sq.shape[1]}", fontsize=9.5)
        ax.axis("off")
    fig_d.suptitle(f"Tahap 2b: Peta Objek Multi-Skala IS-Net (d1..d6 Setelah Sigmoid) — {active_name[:32]}", fontsize=13.5, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.91])
    save_fig(fig_d, "02b_isnet_side_outputs_d1_d6.png")
    plt.show()

# --- 2c. Pascaproses IS-Net & Verifikasi terhadap rembg asli ---
info_m = isnet_active["info_mask"]
mask_is = isnet_active["mask"]
seg_rgb_is = np.where(mask_is[..., None] > 0, img_rgb, WARNA_LATAR).astype(np.uint8)
kotak_is = isnet_active["bbox"]

print("\n=== 2c. PASCAPROSES MASK IS-NET ===")
print("Model segmentasi                  : IS-Net (isnet-general-use, lewat rembg ONNX)")
print(f"Ambang alpha biner                : > {ISNET_ALPHA_THRESH}")
print(f"Area objek pada gambar            : {100.0 * np.mean(mask_is > 0):.2f}%")
print(f"Komponen putih pada mask awal     : {info_m['komponen']}")
print(f"Piksel lubang yang terisi         : {info_m['lubang_terisi']}")
print(f"Piksel noda lepas yang dibuang    : {info_m['noda_dibuang']}")
print(f"Bounding box manual (x, y, w, h)  : {kotak_is}")

try:
    alpha_rembg = np.asarray(session_isnet.predict(Image.fromarray(img_rgb))[0])
    selisih = np.abs(isnet_active["alpha"].astype(np.float32) - alpha_rembg.astype(np.float32)).mean()
    print(f"Verifikasi vs rembg asli (MAE)    : {selisih:.4f} (skala 0-255; mendekati 0 = identik)")
except Exception as exc:
    print("Verifikasi langsung dengan rembg dilewati:", exc)

gambar_kotak_is = img_rgb.copy()
if kotak_is[2] > 0 and kotak_is[3] > 0:
    bx_i, by_i, bw_i, bh_i = kotak_is
    cv2.rectangle(gambar_kotak_is, (bx_i, by_i), (bx_i + bw_i - 1, by_i + bh_i - 1), (40, 220, 80), max(2, img_rgb.shape[1] // 200))

fig_p, axes_p = plt.subplots(1, 6, figsize=(22, 4.0))
axes_p[0].imshow(isnet_active["peta_prob"], cmap="gray");                 axes_p[0].set_title("1. Keluaran d1\n(min-max 0..1)")
axes_p[1].imshow(isnet_active["alpha"], cmap="gray", vmin=0, vmax=255);   axes_p[1].set_title("2. Resize LANCZOS\n(kanal alpha)")
axes_p[2].imshow(isnet_active["mask_raw"], cmap="gray");                  axes_p[2].set_title(f"3. Threshold > {ISNET_ALPHA_THRESH}\n(mask awal)")
axes_p[3].imshow(mask_is, cmap="gray");                                   axes_p[3].set_title("4. Mask akhir\n(kontur maks + isi lubang)")
axes_p[4].imshow(seg_rgb_is);                                             axes_p[4].set_title("5. Hasil segmentasi RGB")
axes_p[5].imshow(gambar_kotak_is);                                        axes_p[5].set_title("6. Bounding box manual")
for ax in axes_p:
    ax.axis("off")
fig_p.suptitle(f"Tahap 2c: Pascaproses IS-Net Menjadi Mask Final — {active_name[:32]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.91])
save_fig(fig_p, "02c_isnet_pascaproses.png")
plt.show()

# --- Ringkasan Visual Segmentasi/Masking IS-Net ---
cov_is = 100.0 * np.mean(mask_is > 0)
fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.2))
axes[0].imshow(img_rgb);                               axes[0].set_title("1. Citra Input RGB")
axes[1].imshow(isnet_active["alpha"], cmap="inferno"); axes[1].set_title("2. Peta Alpha IS-Net")
axes[2].imshow(mask_is, cmap="gray");                  axes[2].set_title(f"3. Mask Biner Bersih\n(area={cov_is:.1f}%)")
axes[3].imshow(make_overlay(img_rgb, mask_is));        axes[3].set_title("4. Overlay Foreground")
for ax in axes:
    ax.axis("off")
fig.suptitle(f"Tahap 2: Segmentasi / Masking IS-Net (isnet-general-use) — {active_name[:36]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.91])
save_fig(fig, "02_segmentasi_isnet.png")
plt.show()

## Cell 5 — Penyempurnaan Mask IS-Net, Kontur & Isolasi Objek

Menyaring kontur hasil masking **IS-Net** berdasarkan rentang luas (`MIN_AREA_RATIO`–`MAX_AREA_RATIO`), menampilkan kontur + *convex hull*, dan mengisolasi objek foreground.

In [ ]:
MIN_AREA_RATIO = 0.01    # Abaikan kontur noise < 1% luas gambar
MAX_AREA_RATIO = 0.95    # Abaikan kontur yang menelan hampir seluruh frame (> 95%)
KEEP_TOP_K_OBJECTS = 1   # Jumlah maksimal objek yang diambil
CHOSEN_LABEL = "IS-Net (isnet-general-use)"


def extract_valid_contours(mask: np.ndarray, min_ratio=0.01, max_ratio=0.95, top_k=1):
    h, w = mask.shape
    img_area = float(h * w)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        ratio = area / img_area
        if min_ratio <= ratio <= max_ratio:
            valid.append((area, cnt))
    valid.sort(key=lambda item: item[0], reverse=True)
    return [cnt for _, cnt in valid[:top_k]]


chosen_raw = isnet_active["alpha"]
chosen_mask = isnet_active["mask"]
valid_contours = extract_valid_contours(chosen_mask, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)

if not valid_contours:
    print("PERINGATAN: Tidak ada kontur lolos filter area. Mengambil kontur terbesar yang tersedia.")
    all_cnts, _ = cv2.findContours(chosen_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if all_cnts:
        valid_contours = [max(all_cnts, key=cv2.contourArea)]

selected_mask = np.zeros_like(chosen_mask)
if valid_contours:
    cv2.drawContours(selected_mask, valid_contours, -1, 255, thickness=cv2.FILLED)

contour_vis = img_rgb.copy()
for cnt in valid_contours:
    hull = cv2.convexHull(cnt)
    cv2.drawContours(contour_vis, [cnt], -1, (0, 255, 80), 3)
    cv2.drawContours(contour_vis, [hull], -1, (255, 220, 0), 2)

cutout = cv2.bitwise_and(img_rgb, img_rgb, mask=selected_mask)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(chosen_raw, cmap="inferno")
axes[0].set_title("1. Peta Alpha IS-Net")
axes[1].imshow(selected_mask, cmap="gray")
axes[1].set_title(f"2. Mask Final IS-Net\n(area={100*np.mean(selected_mask>0):.1f}%)")
axes[2].imshow(contour_vis)
axes[2].set_title("3. Kontur (Hijau) &\nConvex Hull (Kuning)")
axes[3].imshow(cutout)
axes[3].set_title("4. Isolasi Objek Foreground")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Penyempurnaan Mask & Isolasi Objek ({CHOSEN_LABEL}) — {active_name[:34]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.91])
save_fig(fig, "03_mask_kontur_isolasi.png")
plt.show()

## Cell 6 — Tahap 3: Ekstraksi Fitur (Geometri Kontur, HOG, Tekstur GLCM, & Warna)

Mengubah objek kendaraan hasil segmentasi/masking **IS-Net** menjadi **vektor/tabel angka** dan **visualisasi fitur**:
1. **Fitur Bentuk / Geometri**: `Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio (w/h)`, `Extent`, `Solidity`, `Circularity`, dan `Hu Moments`.
2. **Fitur Gradien Bentuk (HOG)**: menangkap pola orientasi tepi bodi, kaca, dan roda kendaraan pada ROI hasil masking IS-Net.
3. **Fitur Tekstur (GLCM)**: `Contrast`, `Dissimilarity`, `Homogeneity`, `Energy`, dan `Correlation`.
4. **Ringkasan Warna (RGB & HSV)** pada area foreground.

In [ ]:
def extract_all_features(rgb: np.ndarray, gray: np.ndarray, mask: np.ndarray, cnt: np.ndarray, obj_id: int = 1):
    area = float(cv2.contourArea(cnt))
    perimeter = float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_area = float(max(1, w * h))
    aspect_ratio = w / float(max(1, h))
    extent = area / bbox_area
    hull = cv2.convexHull(cnt)
    hull_area = float(max(1.0, cv2.contourArea(hull)))
    solidity = area / hull_area
    circularity = (4.0 * math.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0

    hu = cv2.HuMoments(cv2.moments(cnt)).flatten()
    hu_log = [-math.copysign(1.0, val) * math.log10(abs(val)) if abs(val) > 1e-12 else 0.0 for val in hu[:3]]

    roi_rgb = rgb[y : y + h, x : x + w]
    roi_gray = gray[y : y + h, x : x + w]
    roi_mask = mask[y : y + h, x : x + w]

    # Masking latar menjadi abu-abu netral (127) di luar siluet IS-Net sebelum HOG/GLCM
    roi_gray_fg = np.where(roi_mask > 0, roi_gray, 127).astype(np.uint8)
    roi_128 = cv2.resize(roi_gray_fg, (128, 128), interpolation=cv2.INTER_AREA)
    hog_vec, hog_vis = hog(
        roi_128,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=True,
    )

    roi_q = (roi_128 // 8).astype(np.uint8)
    glcm = graycomatrix(
        roi_q,
        distances=[1, 2],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=32,
        symmetric=True,
        normed=True,
    )
    glcm_props = {
        prop: float(np.mean(graycoprops(glcm, prop)))
        for prop in ("contrast", "dissimilarity", "homogeneity", "energy", "correlation")
    }

    fg_rgb = roi_rgb[roi_mask > 0]
    if fg_rgb.size == 0:
        fg_rgb = roi_rgb.reshape(-1, 3)
    hsv_roi = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2HSV)
    fg_hsv = hsv_roi[roi_mask > 0]
    if fg_hsv.size == 0:
        fg_hsv = hsv_roi.reshape(-1, 3)

    row = {
        "Objek": obj_id,
        "BBox_x": x,
        "BBox_y": y,
        "BBox_w": w,
        "BBox_h": h,
        "Area_px2": round(area, 1),
        "Perimeter_px": round(perimeter, 1),
        "Aspect_Ratio": round(aspect_ratio, 3),
        "Extent": round(extent, 3),
        "Solidity": round(solidity, 3),
        "Circularity": round(circularity, 3),
        "Hu1_log": round(hu_log[0], 3),
        "Hu2_log": round(hu_log[1], 3),
        "HOG_Dimensi": int(hog_vec.size),
        "GLCM_Contrast": round(glcm_props["contrast"], 3),
        "GLCM_Dissimilarity": round(glcm_props["dissimilarity"], 3),
        "GLCM_Homogeneity": round(glcm_props["homogeneity"], 3),
        "GLCM_Energy": round(glcm_props["energy"], 3),
        "GLCM_Correlation": round(glcm_props["correlation"], 3),
        "RGB_Mean": f"({fg_rgb[:,0].mean():.0f}, {fg_rgb[:,1].mean():.0f}, {fg_rgb[:,2].mean():.0f})",
        "HSV_Sat_Mean": round(float(fg_hsv[:, 1].mean()), 1),
    }
    artifacts = {
        "bbox": (x, y, w, h),
        "roi_rgb": roi_rgb,
        "roi_128": roi_128,
        "hog_vec": hog_vec,
        "hog_vis": hog_vis,
        "glcm_props": glcm_props,
        "fg_rgb": fg_rgb,
    }
    return row, artifacts


if not valid_contours:
    raise RuntimeError(f"Tidak ada objek terdeteksi pada gambar aktif: {active_name}")

feature_rows = []
feature_artifacts = []
for idx, cnt in enumerate(valid_contours, start=1):
    row, art = extract_all_features(img_rgb, img_gray, selected_mask, cnt, obj_id=idx)
    feature_rows.append(row)
    feature_artifacts.append(art)

df_features = pd.DataFrame(feature_rows)
df_features.to_csv(OUT_DIR / "04_tabel_fitur.csv", index=False)

print("=== TABEL HASIL EKSTRAKSI FITUR (BENTUK, HOG, GLCM, WARNA) ===")
display(df_features.T.rename(columns=lambda c: f"Objek #{c + 1}"))

art0 = feature_artifacts[0]
print(f"\nDimensi vektor fitur HOG : {art0['hog_vec'].size} angka")
print("10 nilai pertama vektor HOG:", np.round(art0["hog_vec"][:10], 4))

fig = plt.figure(figsize=(16, 8.2))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.22)

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(art0["roi_rgb"])
bx, by, bw, bh = art0["bbox"]
ax1.set_title(f"1. ROI Objek Hasil IS-Net\n({bw}x{bh} px | AR={feature_rows[0]['Aspect_Ratio']})")
ax1.axis("off")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(art0["hog_vis"], cmap="inferno")
ax2.set_title(f"2. Visualisasi Fitur HOG\n(Vektor = {art0['hog_vec'].size} dimensi)")
ax2.axis("off")

ax3 = fig.add_subplot(gs[0, 2])
geom_labels = ["Aspect\nRatio", "Extent", "Solidity", "Circularity"]
geom_vals = [
    feature_rows[0]["Aspect_Ratio"],
    feature_rows[0]["Extent"],
    feature_rows[0]["Solidity"],
    feature_rows[0]["Circularity"],
]
bars = ax3.bar(geom_labels, geom_vals, color=["#4c72b0", "#55a868", "#c44e52", "#8172b2"])
for bar, val in zip(bars, geom_vals):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.2f}", ha="center", fontsize=9, fontweight="bold")
ax3.set_title("3. Fitur Geometri / Bentuk")
ax3.set_ylim(0, max(geom_vals) * 1.25 + 0.1)
ax3.grid(axis="y", alpha=0.25)

ax4 = fig.add_subplot(gs[1, 0:2])
glcm_keys = list(art0["glcm_props"].keys())
glcm_vals = [art0["glcm_props"][k] for k in glcm_keys]
bars_g = ax4.barh(glcm_keys, glcm_vals, color="#dd8452")
for bar, val in zip(bars_g, glcm_vals):
    ax4.text(val + max(glcm_vals) * 0.02, bar.get_y() + bar.get_height() / 2, f"{val:.3f}", va="center", fontsize=9)
ax4.set_title("4. Fitur Tekstur GLCM (Gray-Level Co-occurrence Matrix)")
ax4.set_xlim(0, max(glcm_vals) * 1.22 + 0.1)
ax4.grid(axis="x", alpha=0.25)

ax5 = fig.add_subplot(gs[1, 2])
for ch, col, lbl in zip(range(3), ["red", "green", "blue"], ["R", "G", "B"]):
    ax5.hist(art0["fg_rgb"][:, ch], bins=32, range=(0, 256), color=col, alpha=0.45, label=lbl)
ax5.set_title("5. Histogram Warna RGB Foreground\n(Kurang diskriminatif utk jenis kendaraan)")
ax5.legend()
ax5.grid(alpha=0.25)

fig.suptitle(f"Tahap 3: Ekstraksi Fitur (Segmentasi IS-Net) — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "04_ekstraksi_fitur.png")
plt.show()

## Cell 7 — Tahap 4: Hasil Akhir Bounding Box Manual (Tanpa Label) & Diagram Alur 1-Baris (Siap PPT)

Menampilkan **Bounding Box manual tanpa label kelas** dari mask IS-Net (`cv2.boundingRect`) dan **ringkasan alur 5-panel horizontal** siap tempel ke slide PowerPoint.

In [ ]:
SHOW_SIZE_TEXT = False  # Set True jika ingin menampilkan ukuran piksel (tetap tanpa label kelas)

bbox_vis = img_rgb.copy()
for art in feature_artifacts:
    x, y, w, h = art["bbox"]
    cv2.rectangle(bbox_vis, (x, y), (x + w, y + h), (40, 220, 80), 4)
    if SHOW_SIZE_TEXT:
        cv2.putText(
            bbox_vis, f"{w}x{h} px", (x + 4, max(22, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 220, 80), 2, cv2.LINE_AA,
        )

fig1, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(bbox_vis)
ax.set_title(f"Hasil Akhir Segmentasi IS-Net: Bounding Box Manual (Tanpa Label)\nMetode: {CHOSEN_LABEL}")
ax.axis("off")
save_fig(fig1, "05_hasil_bounding_box.png")
plt.show()

fig2, axes = plt.subplots(1, 5, figsize=(20, 4.3))
summary_panels = [
    ("1. Input Citra", img_rgb, None),
    ("2. Pre-processing IS-Net\n(Resize 1024x1024)", pil_kecil, None),
    ("3. Segmentasi / Masking\n(IS-Net General-Use)", selected_mask, "gray"),
    ("4. Ekstraksi Fitur\n(HOG Gradien ROI)", feature_artifacts[0]["hog_vis"], "inferno"),
    ("5. Output Akhir\n(Bounding Box Manual)", bbox_vis, None),
]
for ax, (title, im, cmap) in zip(axes, summary_panels):
    ax.imshow(im, cmap=cmap)
    ax.set_title(title, fontsize=10.5)
    ax.axis("off")

fig2.suptitle(f"Ringkasan Alur Segmentasi IS-Net — {active_name[:36]}", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.90])
save_fig(fig2, "06_alur_lengkap_pcv_isnet.png")
plt.show()

## Cell 8 — Proses Seluruh Gambar (Paralel Penuh RAM & GPU VRAM) + Tampilkan 4 Sampel Acak + Ekspor Semua Output

Memproses **seluruh gambar** di folder secara paralel menggunakan **segmentasi/masking IS-Net** tanpa aturan `if-else` dan tanpa pembatasan RAM maupun VRAM GPU, menampilkan **4 gambar acak** sebagai preview visual di Colab, dan menyimpan **semua** hasilnya (format 100% kompatibel dengan `5_inferensi_ssd_dari_output_nb1.ipynb`):
1. **`per_gambar/0001_<nama>.bbox.jpg`** — semua gambar resolusi asli dengan **bounding box tanpa label kelas**.
2. **`per_gambar/0001_<nama>.json`** — detail koordinat bounding box (piksel & ternormalisasi `0..1`), status `label: null`, info mask IS-Net, dan fitur bentuk/GLCM per gambar.
3. **`07_ringkasan_segmentasi_isnet.csv`** — tabel rekap CSV untuk seluruh gambar.
4. **`07_hasil_bounding_box_semua.json`** — file JSON gabungan seluruh gambar.
5. **`nb1b_pcv_isnet_outputs.zip`** — seluruh grafik PPT + folder `per_gambar/` + CSV & JSON dalam satu paket siap unduh (otomatis disalin juga ke `MyDrive/ferqueue/output_pcv_isnet/` jika Google Drive ter-mount).

In [ ]:
# Pengaturan kecepatan (tanpa batas RAM/VRAM — manfaatkan penuh CPU & GPU Colab)
NUM_WORKERS = max(4, os.cpu_count() or 4)
ISNET_BATCH_SIZE = 8 if "CUDAExecutionProvider" in session_isnet.inner_session.get_providers() else 1
CHUNK_SIZE = 64  # Jumlah gambar yang diprefetch & diproses paralel ke RAM per putaran

PER_IMAGE_DIR = OUT_DIR / "per_gambar"
if PER_IMAGE_DIR.exists():
    shutil.rmtree(PER_IMAGE_DIR)
PER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Cek apakah input batch IS-Net ONNX dinamis (mendukung batch > 1) atau statis (batch = 1)
_in_shape = session_isnet.inner_session.get_inputs()[0].shape
_supports_dynamic_batch = bool(_in_shape) and (isinstance(_in_shape[0], str) or _in_shape[0] is None)
EFFECTIVE_BATCH_SIZE = ISNET_BATCH_SIZE if _supports_dynamic_batch else 1

total_imgs = len(image_paths)
multi_summary = []
all_json_records = []
failed_files = []
sample_vis_cache = {}

t_start = time.perf_counter()
print(
    f"Memulai pemrosesan {total_imgs} gambar (Segmentasi/Masking IS-Net) | "
    f"Provider={session_isnet.inner_session.get_providers()[0]} | "
    f"Batch={EFFECTIVE_BATCH_SIZE} | Workers={NUM_WORKERS} | Chunk={CHUNK_SIZE}"
)


def _load_and_prep_item(args):
    r_idx, img_path = args
    try:
        rgb_i = load_rgb(img_path)
        gray_i = cv2.cvtColor(rgb_i, cv2.COLOR_RGB2GRAY)
        _, _, tensor_i = praproses_isnet(rgb_i)
        return {"ok": True, "r": r_idx, "path": img_path, "rgb": rgb_i, "gray": gray_i, "tensor": tensor_i}
    except Exception as exc:
        return {"ok": False, "r": r_idx, "path": img_path, "error": str(exc)}


def _run_isnet_chunk(valid_items: list):
    """Jalankan inferensi IS-Net secara batch (bila model mendukung) dengan fallback aman ke batch=1."""
    global EFFECTIVE_BATCH_SIZE
    outputs_d1 = [None] * len(valid_items)
    idx_pos = 0
    while idx_pos < len(valid_items):
        bs = min(EFFECTIVE_BATCH_SIZE, len(valid_items) - idx_pos)
        sub = valid_items[idx_pos : idx_pos + bs]
        if bs > 1:
            try:
                batch_tensor = np.concatenate([it["tensor"] for it in sub], axis=0)
                batch_out = jalankan_isnet(batch_tensor, session_isnet)[0]
                for j in range(bs):
                    outputs_d1[idx_pos + j] = batch_out[j : j + 1]
                idx_pos += bs
                continue
            except Exception:
                # Jika graf ONNX menolak batch > 1, turunkan otomatis ke batch = 1
                EFFECTIVE_BATCH_SIZE = 1
        for j, it in enumerate(sub):
            outputs_d1[idx_pos + j] = jalankan_isnet(it["tensor"], session_isnet)[0]
        idx_pos += bs
    return outputs_d1


def _finalize_item(args):
    it, d1_out = args
    r_idx = it["r"]
    img_path = it["path"]
    name = img_path.name
    stem = img_path.stem
    rgb_i = it["rgb"]
    gray_i = it["gray"]
    img_h, img_w = rgb_i.shape[:2]

    try:
        _, alpha_i, mask_raw_i = peta_ke_mask(d1_out, (img_w, img_h), alpha_thresh=ISNET_ALPHA_THRESH)
        mask_is_i, _, isnet_info_i = perbaiki_mask(mask_raw_i)

        cnts_i = extract_valid_contours(mask_is_i, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)
        if not cnts_i:
            all_c_i, _ = cv2.findContours(mask_is_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if all_c_i:
                cnts_i = [max(all_c_i, key=cv2.contourArea)]

        mask_i = np.zeros_like(mask_is_i)
        vis_i = rgb_i.copy()

        objects_json = []
        if cnts_i:
            cv2.drawContours(mask_i, cnts_i, -1, 255, thickness=cv2.FILLED)
            for obj_idx, cnt in enumerate(cnts_i, start=1):
                feat_row, feat_art = extract_all_features(rgb_i, gray_i, mask_i, cnt, obj_id=obj_idx)
                x, y, w, h = feat_art["bbox"]
                x2, y2 = x + w, y + h
                cv2.rectangle(vis_i, (x, y), (x2, y2), (40, 220, 80), 4)
                objects_json.append({
                    "object_id": obj_idx,
                    "label": None,
                    "bbox_xywh_px": [int(x), int(y), int(w), int(h)],
                    "bbox_xyxy_px": [int(x), int(y), int(x2), int(y2)],
                    "bbox_norm_xyxy": [
                        round(x / float(img_w), 6),
                        round(y / float(img_h), 6),
                        round(x2 / float(img_w), 6),
                        round(y2 / float(img_h), 6),
                    ],
                    "bbox_norm_yxyx": [
                        round(y / float(img_h), 6),
                        round(x / float(img_w), 6),
                        round(y2 / float(img_h), 6),
                        round(x2 / float(img_w), 6),
                    ],
                    "area_px2": feat_row["Area_px2"],
                    "aspect_ratio": feat_row["Aspect_Ratio"],
                    "extent": feat_row["Extent"],
                    "solidity": feat_row["Solidity"],
                    "circularity": feat_row["Circularity"],
                    "glcm": {k: round(float(v), 4) for k, v in feat_art["glcm_props"].items()},
                })
            primary = objects_json[0]
            bx, by, bw, bh = primary["bbox_xywh_px"]
            x2_p, y2_p = primary["bbox_xyxy_px"][2], primary["bbox_xyxy_px"][3]
            n_x1, n_y1, n_x2, n_y2 = primary["bbox_norm_xyxy"]
            ar = primary["aspect_ratio"]
            ext = primary["extent"]
            sol = primary["solidity"]
            circ = primary["circularity"]
        else:
            bx = by = bw = bh = x2_p = y2_p = 0
            n_x1 = n_y1 = n_x2 = n_y2 = ar = ext = sol = circ = 0.0

        cov_pct = round(100.0 * float(np.mean(mask_i > 0)), 2)
        bbox_img_name = f"{r_idx + 1:04d}_{stem}.bbox.jpg"
        json_file_name = f"{r_idx + 1:04d}_{stem}.json"
        bbox_img_path = PER_IMAGE_DIR / bbox_img_name
        json_file_path = PER_IMAGE_DIR / json_file_name

        Image.fromarray(vis_i).save(bbox_img_path, format="JPEG", quality=92)

        record_json = {
            "index": r_idx + 1,
            "file_name": name,
            "image_width": int(img_w),
            "image_height": int(img_h),
            "method": "isnet_general_use",
            "segmentation_strategy": "isnet_general_use",
            "isnet_alpha_thresh": int(ISNET_ALPHA_THRESH),
            "isnet_mask_info": isnet_info_i,
            "mask_coverage_pct": cov_pct,
            "object_count": len(objects_json),
            "objects": objects_json,
            "bbox_image_file": f"per_gambar/{bbox_img_name}",
        }
        json_file_path.write_text(json.dumps(record_json, indent=2, ensure_ascii=False), encoding="utf-8")

        summary_row = {
            "No": r_idx + 1,
            "File_Name": name,
            "Image_W": img_w,
            "Image_H": img_h,
            "Metode_Segmentasi": "isnet_general_use",
            "ISNet_Alpha_Thresh": int(ISNET_ALPHA_THRESH),
            "ISNet_Komponen": isnet_info_i["komponen"],
            "ISNet_Lubang_Terisi": isnet_info_i["lubang_terisi"],
            "ISNet_Noda_Dibuang": isnet_info_i["noda_dibuang"],
            "Jumlah_Objek": len(objects_json),
            "Label": "",
            "BBox_xmin": bx,
            "BBox_ymin": by,
            "BBox_xmax": x2_p,
            "BBox_ymax": y2_p,
            "BBox_w": bw,
            "BBox_h": bh,
            "Norm_xmin": n_x1,
            "Norm_ymin": n_y1,
            "Norm_xmax": n_x2,
            "Norm_ymax": n_y2,
            "Aspect_Ratio": ar,
            "Extent": ext,
            "Solidity": sol,
            "Circularity": circ,
            "Cakupan_%": cov_pct,
            "Output_BBox_Image": f"per_gambar/{bbox_img_name}",
            "Output_JSON": f"per_gambar/{json_file_name}",
        }

        sample_vis = None
        if img_path in sample_path_set:
            sample_vis = {
                "name": name,
                "rgb": rgb_i,
                "alpha": alpha_i,
                "mask": mask_i,
                "vis": vis_i,
                "cov_pct": cov_pct,
                "bbox": (bx, by, bw, bh),
            }

        return {"ok": True, "path": img_path, "record": record_json, "summary": summary_row, "sample_vis": sample_vis}
    except Exception as exc:
        return {"ok": False, "path": img_path, "error": str(exc)}


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    for start_idx in range(0, total_imgs, CHUNK_SIZE):
        chunk_args = [(i, image_paths[i]) for i in range(start_idx, min(start_idx + CHUNK_SIZE, total_imgs))]
        loaded_items = list(pool.map(_load_and_prep_item, chunk_args))

        valid_items = []
        for it in loaded_items:
            if it["ok"]:
                valid_items.append(it)
            else:
                failed_files.append({"file_name": it["path"].name, "error": it["error"]})

        if valid_items:
            d1_list = _run_isnet_chunk(valid_items)
            finalized = list(pool.map(_finalize_item, zip(valid_items, d1_list)))
            for res in finalized:
                if res["ok"]:
                    all_json_records.append(res["record"])
                    multi_summary.append(res["summary"])
                    if res["sample_vis"] is not None:
                        sample_vis_cache[res["path"]] = res["sample_vis"]
                else:
                    failed_files.append({"file_name": res["path"].name, "error": res["error"]})

        done = min(start_idx + CHUNK_SIZE, total_imgs)
        elapsed = time.perf_counter() - t_start
        rate = done / max(elapsed, 1e-6)
        eta = (total_imgs - done) / max(rate, 1e-6)
        print(f"  Progres: {done}/{total_imgs} ({100.0 * done / total_imgs:.1f}%) | {rate:.2f} gbr/dtk | {elapsed:.1f} dtk | sisa ~{eta:.0f} dtk")

# Visualisasikan hanya 4 gambar sampel acak di Colab
plot_items = [sample_vis_cache[p] for p in sample_paths if p in sample_vis_cache]
rows_n = len(plot_items)
if rows_n > 0:
    fig, axes = plt.subplots(rows_n, 4, figsize=(16.5, 3.8 * rows_n))
    if rows_n == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, item in enumerate(plot_items):
        bx, by, bw, bh = item["bbox"]
        axes[r, 0].imshow(item["rgb"])
        axes[r, 0].set_title(f"Input: {item['name'][:24]}", fontsize=9)
        axes[r, 1].imshow(item["alpha"], cmap="inferno")
        axes[r, 1].set_title("Alpha Probabilitas IS-Net", fontsize=9)
        axes[r, 2].imshow(make_overlay(item["rgb"], item["mask"]))
        axes[r, 2].set_title(f"Mask IS-Net\narea={item['cov_pct']:.1f}%", fontsize=9)
        axes[r, 3].imshow(item["vis"])
        axes[r, 3].set_title(f"Bounding Box Tanpa Label\n({bx},{by}) {bw}x{bh}", fontsize=9)
        for c in range(4):
            axes[r, c].axis("off")

    fig.suptitle(
        f"Preview {rows_n} Sampel Acak IS-Net (dari Total {len(all_json_records)} Gambar yang Diproses)",
        fontsize=14, fontweight="bold",
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_fig(fig, "07_evaluasi_multi_gambar_isnet.png")
    plt.show()

csv_path = OUT_DIR / "07_ringkasan_segmentasi_isnet.csv"
json_all_path = OUT_DIR / "07_hasil_bounding_box_semua.json"

df_multi = pd.DataFrame(multi_summary)
df_multi.to_csv(csv_path, index=False)
json_all_path.write_text(
    json.dumps(
        {
            "pipeline": "pcv_isnet",
            "description": "Lokalisasi bounding box tanpa label kelas menggunakan segmentasi/masking IS-Net (isnet-general-use).",
            "total_images": len(all_json_records),
            "failed_count": len(failed_files),
            "failed_files": failed_files,
            "results": all_json_records,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"\nSelesai! Berhasil: {len(all_json_records)} | Gagal: {len(failed_files)}")
print(f"Rekap CSV tersimpan      : {csv_path}")
print(f"Rekap JSON gabungan      : {json_all_path}")
print(f"Output per gambar ({len(all_json_records)}x2): {PER_IMAGE_DIR}")
if not df_multi.empty:
    display(df_multi.head(10)[["No", "File_Name", "Metode_Segmentasi", "BBox_xmin", "BBox_ymin", "BBox_w", "BBox_h", "Aspect_Ratio", "Cakupan_%", "Output_BBox_Image"]])

zip_path = OUT_DIR.parent / "nb1b_pcv_isnet_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file() and "_inputs" not in p.relative_to(OUT_DIR).parts:
            zf.write(p, p.relative_to(OUT_DIR).as_posix())
print(f"\nSemua output ({len(all_json_records)} gambar + JSON + CSV + grafik PPT) tersimpan di: {zip_path} ({zip_path.stat().st_size / (1024**2):.1f} MB)")

# Jika Google Drive ter-mount di Colab, salin ZIP + CSV + JSON ke Drive agar mudah diunduh tanpa timeout
drive_out_dir = Path("/content/drive/MyDrive/ferqueue/output_pcv_isnet")
if IN_COLAB and Path("/content/drive/MyDrive").is_dir():
    drive_out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_out_dir / zip_path.name)
    shutil.copy2(csv_path, drive_out_dir / csv_path.name)
    shutil.copy2(json_all_path, drive_out_dir / json_all_path.name)
    print(f"Salinan ZIP, CSV, & JSON juga disimpan ke Google Drive: {drive_out_dir}")

# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP lewat browser Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))